In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter
import fsspec

# ── Load Data ──────────────────────────────────────────────────────────────────
def load_specific_parquet(url):
    with fsspec.open(url, 'rb') as f:
        return pd.read_parquet(f)

url  = "gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro.parquet"
metro_24_25 = load_specific_parquet(url)

#---------------Loading the corridor list and cleaning it -------------------------
flight_route = pd.read_excel("Flight Paris for Size the Prize v1.xlsx")
corridors = flight_route[~flight_route['Row Labels'].str.contains('Unknown', na=False)]['Row Labels'].str.strip().iloc[:-1]
corridors = corridors.str.replace(' - ','->')
# Filter out corridors where From and To are identical
corridors = corridors[corridors.apply(lambda x: x.split('->')[0] != x.split('->')[1])]

Data = metro_24_25.copy()


In [14]:
import pandas as pd
import numpy as np

# ── Precompute constants ───────────────────────────────────────────────────────────────
WU_SUBSIDIARIES = ['Wheels Up Private Jets', 'Wheels Up LLC', 'Mountain Aviation', 'Alante Air Charter']

Thresh = False
JET_TYPE = 'Super Midsize Jet'
DAY_ORDER    = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
BUCKET_ORDER = ['07:00–10:00', '10:00–13:00', '13:00–16:00',
                '16:00–19:00', '19:00–24:00', '00:00–07:00']

_HOUR_TO_BUCKET = np.array(
    ['00:00–07:00'] * 7 +   # hours 0–6
    ['07:00–10:00'] * 3 +   # hours 7–9
    ['10:00–13:00'] * 3 +   # hours 10–12
    ['13:00–16:00'] * 3 +   # hours 13–15
    ['16:00–19:00'] * 3 +   # hours 16–18
    ['19:00–24:00'] * 5     # hours 19–23
)

TIER_BINS   = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, float('inf')]
TIER_LABELS = ['0 - 1', '1 - 2', '2 - 3', '3 - 4', '4 - 5', '5 - 6','6 - 7','7 - 8','8 - 9','9 - 10','10+']  # adjust as needed

# ── Preprocess ONCE before the loop ───────────────────────────────────────────────────
data = Data.iloc[:, 1:].copy()

# filter all the corridors that have same FromMetro and ToMetro
data = data[data['FromMetro'] != data['ToMetro']]

# filtering the Aircraft type 
data = data[data['aircraft_segment'].isin([JET_TYPE])]

# Build route column
data['Route'] = data['FromMetro'] + '->' + data['ToMetro']

# Parse datetime on full dataset once
data['dep_dt']     = pd.to_datetime(data['FlightDate_ET'], errors='coerce')
data['dep_hour']   = data['dep_dt'].dt.hour
data['day_name']   = pd.Categorical(data['dep_dt'].dt.day_name(), categories=DAY_ORDER, ordered=True)
data['hour_bucket']= pd.Categorical(_HOUR_TO_BUCKET[data['dep_hour'].to_numpy()], categories=BUCKET_ORDER, ordered=True)
data['is_WU']      = data['Operator'].isin(WU_SUBSIDIARIES).astype(int)

# ── Step 2: Network-level LI (scalar) ─────────────────────────────────────────────────
all_flights_network    = len(data)                      # (a) all flights in dataset
wu_flights_network     = data['is_WU'].sum()            # (b) WU flights in dataset
Network_WU_LI          = wu_flights_network / all_flights_network   # (c) Network LI = b/a

# ── Step 3: Filter to required corridors only ──────────────────────────────────────────
valid_corridors = list(set(corridors).intersection(data['Route'].unique()))

missing = [cor for cor in corridors if cor not in data['Route'].values]
if missing:
    print(f"Note: The following corridors were not found in data and will be skipped: {missing}")

data_filtered = data[data['Route'].isin(valid_corridors)]

# ── Step 4: Cell-level aggregation (Route + DOW + TOD) ────────────────────────────────
# Each cell = one Route + day_name + hour_bucket combination
cell_group = ['Route', 'day_name', 'hour_bucket']

final_report = (
    data_filtered
    .groupby(cell_group, observed=False)
    .agg(
        total_flights_cell = ('Route', 'size'),          # (b) all flights in this cell
        wu_flights_cell    = ('is_WU', 'sum')            # (a) WU flights in this cell
    )
    .reset_index())

# ── Step 5: Cell-level WU LI = WU flights in cell / all flights in cell ───────────────
# Avoid division by zero for empty cells
final_report['WU_LI_corridor'] = np.where(
    final_report['total_flights_cell'] > 0,
    final_report['wu_flights_cell'] / final_report['total_flights_cell'],   # (c)
    0
)

# ── Step 6: Network LI (repeated as column for reference) ─────────────────────────────
final_report['Network_WU_LI'] = Network_WU_LI

# ── Step 7: Relative LI = WU_LI_corridor / Network_WU_LI ─────────────────────────────
final_report['WU_LI_Relative'] = np.where(
    final_report['Network_WU_LI'] > 0,
    final_report['WU_LI_corridor'] / final_report['Network_WU_LI'],         # (d)
    0
)

# ── Step 8: Percentage of flights in cell vs total corridor flights ────────────────────
final_report['total_flights_corridor'] = (
    final_report.groupby('Route')['total_flights_cell'].transform('sum')
)
final_report['pct_flights_cell'] = (
    final_report['wu_flights_cell'] / final_report['total_flights_corridor'] * 100
)

# ── Step 9: Tier buckets ───────────────────────────────────────────────────────────────
TIER_BINS_FULL   = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, float('inf')]
TIER_LABELS_FULL = ['0 - 1', '1 - 2', '2 - 3', '3 - 4', '4 - 5', '5 - 6','6 - 7','7 - 8','8 - 9','9 - 10','10+']  # adjust as needed

# Tier on WU_LI_Relative  (e)
final_report['WU_LI_gap'] = pd.cut(
    final_report['WU_LI_Relative'],
    bins=TIER_BINS_FULL,
    labels=TIER_LABELS_FULL,
    right=False
)

# Tier on pct of flights (optional, kept from original)
final_report['Tier_gap'] = pd.cut(
    final_report['pct_flights_cell'],
    bins=TIER_BINS_FULL,
    labels=TIER_LABELS_FULL,
    right=False
)

# 1. Convert Tier_gap to string to allow the new 'insufficient' labe
final_report['Tier_gap'] = final_report['Tier_gap'].astype(str)

# 2. Update Tier_gap where flight_count is less than 30
if Thresh:
    final_report.loc[final_report['flight_count'] < Thresh, 'Tier_gap'] = 'insufficient'


In [3]:
# LI_Light_Jet = final_report['WU_LI_gap'].value_counts().reset_index().sort_values(by='WU_LI_gap')
# LI_SMID = final_report['WU_LI_gap'].value_counts().reset_index().sort_values(by='WU_LI_gap')

In [4]:
LI_Light_Jet = final_report

In [15]:
LI_SMID = final_report

In [17]:
# Define the filename
output_file = 'LI_Analysis_LJ_SMID.xlsx'

# Use ExcelWriter to save multiple sheets
with pd.ExcelWriter(output_file) as writer:
    LI_Light_Jet.to_excel(writer, sheet_name='LI Light Jet', index=False)
    LI_SMID.to_excel(writer, sheet_name='LI SMID', index=False)

print(f"Successfully saved to {output_file}")

Successfully saved to LI_Analysis_LJ_SMID.xlsx


In [9]:
# WU_LI_gap_count.to_excel("Wheels_Relative_count.xlsx",index=False)

In [10]:
# final_report.to_excel("Wheels_Relative_LI.xlsx",index=False)

In [11]:
# WU_LI_gap_count = final_report['WU_LI_gap'].value_counts().reset_index()